# Guía de Estudio – Parcial 2 Práctico
## ISIS-2611 | Aprendizaje de Máquina

---

### Estructura del parcial (según el tablero)
1. **Leer el caso** → identificar tipo de problema y arquitectura
2. **Detectar errores** en el código dado
3. **Plantear y/o implementar** la solución correcta
4. **Analizar** resultados

### Temas que entran
| Área | Subtemas |
|------|----------|
| ML clásico | Bayes, K-Means, DBSCAN, Jerárquico |
| Deep Learning | MLP (Feed Forward), Backprop |
| Visión | CNN (Conv2D, Pooling, Flatten) |
| NLP | Embeddings, RNN, LSTM, GRU |
| Evaluación | Métricas correctas, data leakage, overfitting |

---
# PARTE 1 – Referencia rápida: Reglas de oro

## 1.1 Activaciones y pérdidas correctas

| Tarea | Salida | Activación final | Pérdida |
|-------|--------|------------------|---------|
| Clasificación binaria | 1 neurona | `sigmoid` | `binary_crossentropy` |
| Clasificación multiclase (etiquetas enteras) | N neuronas | `softmax` | `sparse_categorical_crossentropy` |
| Clasificación multiclase (one-hot) | N neuronas | `softmax` | `categorical_crossentropy` |
| Regresión | 1 neurona | `linear` / ninguna | `mse` |

## 1.2 Cuándo usar qué modelo de clustering

| Situación | Modelo |
|-----------|--------|
| Clusters esféricos y datos numéricos escalados | K-Means |
| Clusters de forma arbitraria, hay outliers | DBSCAN |
| No sabes cuántos clusters hay, quieres dendrograma | Jerárquico |

## 1.3 Pipeline correcto para NLP
```
Texto crudo → Tokenización → Índices/IDs → Embedding → Red (RNN/LSTM/Dense)
```
❌ Nunca: texto crudo directo a la red.

## 1.4 Pipeline correcto para imágenes (CNN)
```
Imagen → Normalizar (÷255) → Conv2D → MaxPooling → ... → Flatten → Dense → Softmax
```
❌ Nunca: Dense antes de Flatten en una CNN.

## 1.5 Regla del data leakage
```python
# CORRECTO: split primero, fit solo en train
X_train, X_test, y_train, y_test = train_test_split(X, y)
scaler.fit(X_train)          # ← fit SOLO en train
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)

# ERROR: fit en todo el dataset antes del split
scaler.fit(X)                # ← contamina el test set
X_scaled = scaler.transform(X)
X_train, X_test = train_test_split(X_scaled, y)
```

---
# PARTE 2 – Plantillas correctas de referencia

In [ ]:
# ============================================================
# PLANTILLA A: MLP para clasificación binaria
# ============================================================
from tensorflow import keras
from tensorflow.keras import layers

model_binario = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(n_features,)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')   # ← sigmoid para binario
])
model_binario.compile(
    optimizer='adam',
    loss='binary_crossentropy',             # ← pérdida binaria
    metrics=['accuracy']
)

In [ ]:
# ============================================================
# PLANTILLA B: MLP para clasificación multiclase (10 clases)
# ============================================================
model_multi = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),   # ← Flatten si la entrada es 2D
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')  # ← softmax para N clases
])
model_multi.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # ← etiquetas son enteros (0-9)
    metrics=['accuracy']
)

In [ ]:
# ============================================================
# PLANTILLA C: CNN para clasificación de imágenes (CIFAR-10)
# ============================================================
from tensorflow.keras import layers, models

model_cnn = models.Sequential([
    # Bloque convolucional 1
    layers.Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.2),

    # Bloque convolucional 2
    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.2),

    # Clasificador
    layers.Flatten(),                        # ← SIEMPRE antes de Dense
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')   # ← 10 clases
])
model_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# RECUERDA: normalizar las imágenes antes de entrenar
# train_images = train_images / 255.0
# test_images  = test_images  / 255.0

In [ ]:
# ============================================================
# PLANTILLA D: Embedding + LSTM para texto
# ============================================================
VOCAB_SIZE   = 10000   # tamaño del vocabulario
EMBED_DIM    = 64      # dimensión de los embeddings
MAX_LEN      = 200     # longitud máxima de secuencia
NUM_CLASES   = 5       # número de categorías

model_lstm = keras.Sequential([
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN),
    layers.LSTM(64, return_sequences=False),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASES, activation='softmax')
])
model_lstm.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Pipeline correcto de texto:
# texto → tokenización → pad_sequences → Embedding layer → LSTM

In [ ]:
# ============================================================
# PLANTILLA E: K-Means correcto
# ============================================================
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import silhouette_score

# 1. ESCALAR SIEMPRE antes de clustering
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_numericas)   # ← solo variables numéricas

# 2. Elegir k con método del codo
inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

# 3. Confirmar con silhouette
scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    scores.append(silhouette_score(X_scaled, labels))

---
# PARTE 3 – Ejercicios: Encuentra el error

> **Instrucciones**: Lee cada celda de código con errores, identifica QUÉ está mal y POR QUÉ, luego mira la solución en la celda siguiente.

---
## Ejercicio 1 – MLP con activación y pérdida incorrectas

**Caso**: Queremos clasificar correos como spam (1) o no spam (0). El modelo tiene el siguiente código:

In [ ]:
# ❌ CÓDIGO CON ERRORES – Ejercicio 1
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(500,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='relu')       # ERROR 1
])
model.compile(
    optimizer='adam',
    loss='mse',                              # ERROR 2
    metrics=['accuracy']
)

# Pregunta: ¿Cuáles son los 2 errores y cómo se corrigen?

In [ ]:
# ✅ SOLUCIÓN – Ejercicio 1
#
# ERROR 1: activation='relu' en la capa de salida binaria
#   ReLU produce cualquier valor >= 0. Una probabilidad debe estar en [0,1].
#   CORRECCIÓN: activation='sigmoid'
#
# ERROR 2: loss='mse' para clasificación binaria
#   MSE mide distancia numérica. Para clasificación usamos cross-entropy.
#   CORRECCIÓN: loss='binary_crossentropy'

model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(500,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')    # ✅
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',              # ✅
    metrics=['accuracy']
)

---
## Ejercicio 2 – CNN sin Flatten y normalización faltante

**Caso**: Clasificación de imágenes CIFAR-10 (32×32×3, 10 clases).

In [ ]:
# ❌ CÓDIGO CON ERRORES – Ejercicio 2
import tensorflow as tf
from tensorflow.keras import datasets, layers, models

(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()
# train_images tiene rango [0, 255]

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),
    layers.Dense(64, activation='relu'),     # ERROR 1
    layers.Dense(10, activation='sigmoid')  # ERROR 2
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',              # ERROR 3
    metrics=['accuracy']
)

model.fit(train_images, train_labels, epochs=10)   # ERROR 4 (implícito)

In [ ]:
# ✅ SOLUCIÓN – Ejercicio 2
#
# ERROR 1: Dense sin Flatten previo
#   Después de Conv2D/MaxPooling el tensor es 3D (h, w, c).
#   Dense espera 1D. Se necesita Flatten() en el medio.
#
# ERROR 2: activation='sigmoid' para 10 clases
#   Sigmoid es para binario. Para N clases usar softmax.
#
# ERROR 3: loss='binary_crossentropy' para 10 clases
#   Usar sparse_categorical_crossentropy (etiquetas como enteros).
#
# ERROR 4: No se normalizaron las imágenes
#   Los píxeles deben dividirse entre 255 para llevarlos a [0,1].

train_images = train_images / 255.0          # ✅ normalizar
test_images  = test_images  / 255.0

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),                         # ✅ Flatten antes de Dense
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')    # ✅ softmax para 10 clases
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',   # ✅
    metrics=['accuracy']
)

---
## Ejercicio 3 – Data Leakage en preprocesamiento

**Caso**: Se quiere entrenar un modelo para predecir el precio de inmuebles.

In [ ]:
# ❌ CÓDIGO CON ERRORES – Ejercicio 3
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df.drop('Precio', axis=1)
y = df['Precio']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)          # ERROR: fit en todo X

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

In [ ]:
# ✅ SOLUCIÓN – Ejercicio 3
#
# ERROR: scaler.fit_transform(X) antes del split
#   El scaler aprende la media y std de TODOS los datos, incluyendo el test set.
#   Eso filtra información del futuro al modelo → data leakage.
#   El modelo tendrá rendimiento artificialmente inflado.
#
# CORRECCIÓN: siempre split PRIMERO, luego fit solo en train.

X = df.drop('Precio', axis=1)
y = df['Precio']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)  # ✅ split primero

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # ✅ fit SOLO en train
X_test  = scaler.transform(X_test)        # ✅ transform (no fit) en test

---
## Ejercicio 4 – K-Means sin escalado y con variables incorrectas

**Caso**: Segmentación de clientes con variables mixtas.

In [ ]:
# ❌ CÓDIGO CON ERRORES – Ejercicio 4
from sklearn.cluster import KMeans

# df tiene columnas: 'edad', 'ingresos', 'ciudad', 'num_compras'
# 'ciudad' es categórica (texto)

X = df[['edad', 'ingresos', 'ciudad', 'num_compras']]   # ERROR 1

modelo = KMeans(n_clusters=3)   # ERROR 2 (metodológico)
modelo.fit(X)                   # ERROR 3 (implícito)

# Se evalúa con accuracy
# accuracy_score(y_real, modelo.labels_)  # ERROR 4

In [ ]:
# ✅ SOLUCIÓN – Ejercicio 4
#
# ERROR 1: incluir variable categórica 'ciudad' directamente
#   K-Means usa distancias Euclidianas → no funciona con texto.
#   Solución: eliminarla o codificarla (con cuidado, puede distorsionar).
#
# ERROR 2: elegir k=3 sin justificación
#   Se debe usar método del codo (inercia) + silhouette score.
#
# ERROR 3: no escalar los datos
#   'ingresos' puede ser 50.000–5.000.000 y 'edad' 18–80.
#   La distancia estará dominada por ingresos. Escalar primero.
#
# ERROR 4: usar accuracy en clustering NO supervisado
#   No hay etiquetas reales. Usar silhouette_score o inercia.

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import silhouette_score

X = df[['edad', 'ingresos', 'num_compras']]   # ✅ solo numéricas

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)            # ✅ escalar

# ✅ elegir k con evidencia
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    print(f'k={k} → silhouette={score:.3f}')

---
## Ejercicio 5 – Embedding y LSTM para NLP

**Caso**: Clasificación de sentimientos de reseñas de películas (positivo/negativo).

In [ ]:
# ❌ CÓDIGO CON ERRORES – Ejercicio 5
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

# textos = ['I loved this movie', 'Terrible film', ...]
# etiquetas = [1, 0, ...]

# Se pasan los textos DIRECTAMENTE como entrada
X = np.array(textos)   # ERROR 1: texto crudo, no numérico

model = keras.Sequential([
    layers.LSTM(64),                              # ERROR 2: no hay Embedding
    layers.Dense(1, activation='softmax')        # ERROR 3: softmax para binario
])
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',             # ERROR 4
    metrics=['accuracy']
)

model.fit(X, etiquetas, epochs=5)

In [ ]:
# ✅ SOLUCIÓN – Ejercicio 5
#
# ERROR 1: texto crudo a la red
#   Las redes solo procesan números. Hay que tokenizar y convertir a secuencias.
#
# ERROR 2: LSTM sin capa de Embedding
#   El embedding convierte índices enteros en vectores densos aprendibles.
#   Sin embedding, la red no puede procesar texto correctamente.
#
# ERROR 3: softmax para clasificación binaria
#   Para 1 sola neurona de salida usar sigmoid.
#
# ERROR 4: categorical_crossentropy para binario
#   Usar binary_crossentropy.

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 10000
MAX_LEN    = 200

# ✅ Tokenización
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(textos_train)         # ← fit solo en train

X_train_seq = tokenizer.texts_to_sequences(textos_train)
X_test_seq  = tokenizer.texts_to_sequences(textos_test)

# ✅ Padding para igualar longitudes
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post')

# ✅ Arquitectura correcta
model = keras.Sequential([
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=64, input_length=MAX_LEN),
    layers.LSTM(64),
    layers.Dense(1, activation='sigmoid')    # ✅ sigmoid para binario
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',              # ✅
    metrics=['accuracy']
)

---
## Ejercicio 6 – Naive Bayes: métrica incorrecta en dataset desbalanceado

**Caso**: Detección de fraude. 98% transacciones normales, 2% fraudes.

In [ ]:
# ❌ CÓDIGO CON ERRORES – Ejercicio 6
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

bayes = GaussianNB()
bayes.fit(X_train, y_train)
y_pred = bayes.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.2f}')  # Imprime 0.98
# El equipo celebra... pero el modelo nunca detecta fraudes

# Pregunta: ¿qué está mal?

In [ ]:
# ✅ SOLUCIÓN – Ejercicio 6
#
# ERROR: usar accuracy en dataset desbalanceado
#   Un modelo que predice SIEMPRE 'no fraude' obtiene 98% accuracy
#   pero no detecta ni un solo fraude real. Es inútil para el negocio.
#
# CORRECCIÓN: usar métricas que consideren el desbalance:
#   - Precision, Recall, F1-score por clase
#   - ROC-AUC
#   - Matriz de confusión

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = bayes.predict(X_test)
y_prob = bayes.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicho'); plt.ylabel('Real')
plt.show()

# Si el recall de 'Fraude' es muy bajo → el modelo falla donde más importa

---
## Ejercicio 7 – Regresión con pérdida de clasificación

**Caso**: Predecir el precio de una casa en millones de pesos (valor continuo).

In [ ]:
# ❌ CÓDIGO CON ERRORES – Ejercicio 7
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='softmax')        # ERROR 1
])
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',             # ERROR 2
    metrics=['accuracy']                         # ERROR 3
)

In [ ]:
# ✅ SOLUCIÓN – Ejercicio 7
#
# ERROR 1: activation='softmax' en regresión
#   Softmax produce una distribución de probabilidad que suma 1.
#   En regresión necesitamos un valor continuo sin restricción.
#   CORRECCIÓN: sin activación (o 'linear')
#
# ERROR 2: categorical_crossentropy para regresión
#   CORRECCIÓN: loss='mse' (Mean Squared Error)
#
# ERROR 3: accuracy no tiene sentido en regresión
#   CORRECCIÓN: metrics=['mae'] (Mean Absolute Error)

model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)                              # ✅ sin activación para regresión
])
model.compile(
    optimizer='adam',
    loss='mse',                                  # ✅
    metrics=['mae']                              # ✅
)

---
## Ejercicio 8 – DBSCAN: entender parámetros

**Caso**: Se quiere agrupar ubicaciones de incidentes en una ciudad. Los datos tienen outliers (puntos aislados).

In [ ]:
# ❌ CÓDIGO CON ERRORES – Ejercicio 8
from sklearn.cluster import DBSCAN

# Datos sin escalar, coordenadas GPS en grados (lat/lon ~ 4.6, -74.1)
X = df[['latitud', 'longitud']]

dbscan = DBSCAN(eps=10, min_samples=5)   # ERROR: eps en unidades sin escalar
labels = dbscan.fit_predict(X)

print(f'Clusters encontrados: {len(set(labels)) - (1 if -1 in labels else 0)}')
# Resultado: 0 clusters (todos son ruido o 1 cluster gigante)

In [ ]:
# ✅ SOLUCIÓN – Ejercicio 8
#
# ERROR: eps=10 es enorme en coordenadas geográficas reales (grados).
#   Con lat/lon, 10 grados = miles de kilómetros → un solo cluster.
#   También: los datos no están escalados.
#
# CONCEPTOS CLAVE de DBSCAN:
#   - eps: radio de vecindad (todos los puntos a distancia ≤ eps son vecinos)
#   - min_samples: mínimo de puntos para formar un cluster core
#   - label == -1 → el punto es RUIDO (outlier)
#   - NO hay que especificar el número de clusters de antemano
#   - Funciona bien con formas arbitrarias y maneja outliers nativamente

from sklearn.preprocessing import StandardScaler

X = df[['latitud', 'longitud']]
X_scaled = StandardScaler().fit_transform(X)    # ✅ escalar

# eps debe calibrarse con k-distance plot, pero valores ~0.3-0.5 son un punto de partida
dbscan = DBSCAN(eps=0.4, min_samples=5)
labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = list(labels).count(-1)
print(f'Clusters: {n_clusters}, Outliers (ruido): {n_noise}')

---
## Ejercicio 9 – CNN: shape incorrecto de entrada

**Caso**: Imágenes en escala de grises (28×28) para clasificar dígitos.

In [ ]:
# ❌ CÓDIGO CON ERRORES – Ejercicio 9
# X_train.shape = (60000, 28, 28)  ← sin canal de color

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28, 28)),  # ERROR 1
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

model.fit(X_train, y_train, epochs=5)  # ERROR 2: X_train sin reshape

In [ ]:
# ✅ SOLUCIÓN – Ejercicio 9
#
# ERROR 1: input_shape=(28, 28) — falta el canal
#   Conv2D espera (height, width, channels).
#   Para escala de grises: channels=1 → input_shape=(28, 28, 1)
#
# ERROR 2: X_train tiene shape (60000, 28, 28) sin canal
#   Hay que hacer reshape para agregar la dimensión del canal.

X_train = X_train.reshape(-1, 28, 28, 1) / 255.0   # ✅ añadir canal + normalizar
X_test  = X_test.reshape(-1, 28, 28, 1)  / 255.0

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28, 28, 1)),  # ✅
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

---
## Ejercicio 10 – Bayes generativo vs discriminativo

**Pregunta conceptual** (responde en texto):

1. ¿Por qué Naive Bayes es un modelo **generativo**?
2. ¿En qué se diferencia de un modelo **discriminativo** como Regresión Logística?
3. ¿Qué significa que Naive Bayes puede **generar** nuevas muestras?

**✅ RESPUESTA – Ejercicio 10**

**1. Por qué Bayes es generativo:**
- Aprende la distribución conjunta P(X, Y) = P(X|Y) · P(Y)
- Aprende CÓMO son los datos de cada clase (media y varianza de cada feature por clase)
- Puede responder: "dado que es spam, ¿cómo se ven las palabras?"

**2. Diferencia con discriminativo:**
- Discriminativo (Regresión Logística, SVM, Red Neuronal) aprende directamente P(Y|X)
- Solo aprende la frontera de decisión, no la distribución de los datos
- No puede generar nuevas muestras
- Suele tener mejor accuracy en clasificación cuando hay suficientes datos

**3. Generación con Bayes:**
- En la práctica del curso se vio cómo Naive Bayes generaba imágenes de dígitos:
  ```python
  mean = bayes.theta_[clase, :]   # media aprendida de cada pixel
  var  = bayes.var_[clase, :]     # varianza aprendida
  muestra = np.random.normal(mean, np.sqrt(var))
  ```
- Porque conoce la distribución P(X|Y=clase), puede samplear nuevas instancias

---
## Ejercicio 11 – RNN vs LSTM: cuándo usar cada uno

In [ ]:
# ❌ CÓDIGO CON ERRORES – Ejercicio 11
# Caso: análisis de sentimientos en reseñas largas (200+ tokens)

model = keras.Sequential([
    layers.Embedding(10000, 64, input_length=200),
    layers.SimpleRNN(64),              # ERROR: RNN simple para secuencias largas
    layers.Dense(1, activation='sigmoid')
])

# Pregunta: ¿Cuál es el problema y cuándo usar LSTM o GRU?

In [ ]:
# ✅ SOLUCIÓN – Ejercicio 11
#
# ERROR: SimpleRNN en secuencias largas
#   SimpleRNN sufre de vanishing gradient: con 200+ pasos, el gradiente
#   se desvanece y la red 'olvida' el contexto del inicio de la secuencia.
#   Para textos largos donde el contexto distante importa → LSTM o GRU.
#
# CUÁNDO USAR CADA UNO:
#   SimpleRNN → secuencias muy cortas (< 30 pasos), contexto inmediato
#   LSTM      → secuencias largas, dependencias a largo plazo
#   GRU       → como LSTM pero más rápido, similar rendimiento, menos parámetros

# ✅ Con LSTM
model_lstm = keras.Sequential([
    layers.Embedding(10000, 64, input_length=200),
    layers.LSTM(64),
    layers.Dense(1, activation='sigmoid')
])

# ✅ Con GRU (alternativa, generalmente más rápida)
model_gru = keras.Sequential([
    layers.Embedding(10000, 64, input_length=200),
    layers.GRU(64),
    layers.Dense(1, activation='sigmoid')
])

---
## Ejercicio 12 – Caso completo: detectar TODOS los errores

**Caso**: Clasificar tickets de soporte bancario en 5 categorías usando texto en español.

In [ ]:
# ❌ CÓDIGO CON MÚLTIPLES ERRORES – Ejercicio 12
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

df = pd.read_csv('tickets_bancarios.csv')

# Preprocesamiento
scaler = StandardScaler()
X_texto = scaler.fit_transform(df['texto'])     # ERROR A: StandardScaler a texto

X_train, X_test, y_train, y_test = train_test_split(
    X_texto, df['categoria'],
    test_size=0.2
)                                                # ERROR B: split después de 'escalar'

# Modelo
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(1,)),   # ERROR C: input_shape=(1,) para texto
    layers.Dense(5, activation='relu')           # ERROR D: relu en salida multiclase
])
model.compile(
    loss='mse',                                  # ERROR E
    optimizer='adam',
    metrics=['accuracy']
)
model.fit(X_train, y_train, epochs=10)

In [ ]:
# ✅ SOLUCIÓN COMPLETA – Ejercicio 12
#
# ERROR A: StandardScaler a texto → No tiene sentido. Texto no es numérico.
#   SOLUCIÓN: tokenizar y usar embeddings (o TF-IDF para enfoques clásicos).
#
# ERROR B: leakage (aunque acá el 'scaler' en texto ni aplica, el orden es incorrecto)
#   SOLUCIÓN: split primero, tokenizar/fit solo en train.
#
# ERROR C: input_shape=(1,) para texto
#   Después de tokenizar y pad, la entrada es (MAX_LEN,) índices de tokens.
#   Para embedding se empieza con la capa Embedding, no Dense.
#
# ERROR D: relu en salida multiclase (5 clases)
#   SOLUCIÓN: softmax
#
# ERROR E: loss='mse' para clasificación multiclase
#   SOLUCIÓN: sparse_categorical_crossentropy (si etiquetas son enteros)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

VOCAB_SIZE = 5000
MAX_LEN    = 128
N_CLASES   = 5

# 1. Split PRIMERO
X_raw = df['texto'].tolist()
le = LabelEncoder()
y = le.fit_transform(df['categoria'])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Tokenizar solo con train
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_raw)                         # ✅ solo train

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_raw), maxlen=MAX_LEN)
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test_raw),  maxlen=MAX_LEN)

# 3. Modelo correcto
model = keras.Sequential([
    layers.Embedding(VOCAB_SIZE, 64, input_length=MAX_LEN), # ✅ Embedding
    layers.LSTM(64),
    layers.Dense(N_CLASES, activation='softmax')             # ✅ softmax
])
model.compile(
    loss='sparse_categorical_crossentropy',                  # ✅
    optimizer='adam',
    metrics=['accuracy']
)
model.fit(X_train_seq, y_train, epochs=10, validation_split=0.1)

---
# PARTE 4 – Resumen: Errores más frecuentes por categoría

## 4.1 MLP / Feed Forward
| Error | Síntoma | Corrección |
|-------|---------|------------|
| `relu` en salida binaria | Salidas > 1, loss no converge | `sigmoid` |
| `relu` en salida multiclase | No es distribución de prob | `softmax` |
| `mse` para clasificación | Loss baja pero accuracy mala | `binary_crossentropy` o `sparse_categorical_crossentropy` |
| No normalizar entradas | Convergencia lenta / inestable | Dividir por 255 (imágenes) o usar `StandardScaler` |
| Data leakage en scaler | Rendimiento inflado en test | Split primero, fit solo en train |

## 4.2 CNN
| Error | Síntoma | Corrección |
|-------|---------|------------|
| `Dense` antes de `Flatten` | Error de shape en el modelo | Agregar `Flatten()` antes |
| `input_shape` sin canal | Error al construir el modelo | `(H, W, C)` — ej: `(32,32,3)` o `(28,28,1)` |
| No normalizar imágenes `÷255` | Gradientes explosivos | `images / 255.0` |
| Loss incorrecta | Modelo no aprende | `sparse_categorical_crossentropy` para multiclase |

## 4.3 Embeddings / NLP
| Error | Síntoma | Corrección |
|-------|---------|------------|
| Texto crudo a la red | Error de tipo | Tokenizar + `pad_sequences` |
| LSTM sin Embedding | No procesa texto correctamente | Capa `Embedding` antes del LSTM |
| `fit` tokenizer en todo el corpus | Leakage | `fit_on_texts(X_train)` solo |
| SimpleRNN en secuencias largas | Vanishing gradient | Usar `LSTM` o `GRU` |

## 4.4 Clustering
| Error | Síntoma | Corrección |
|-------|---------|------------|
| No escalar antes de K-Means | Clusters dominados por variables grandes | `RobustScaler` o `StandardScaler` |
| Variables categóricas directas | K-Means no entiende distancias | Eliminar o codificar con cuidado |
| k elegido sin evidencia | Clusters arbitrarios | Método del codo + silhouette |
| Accuracy para evaluar clustering | No tiene sentido | `silhouette_score`, inercia |
| eps muy grande en DBSCAN | Un solo cluster o todo ruido | Calibrar con k-distance plot |

## 4.5 Bayes
| Error | Síntoma | Corrección |
|-------|---------|------------|
| Accuracy en dataset desbalanceado | Parece perfecto pero no detecta la clase minoritaria | F1, ROC-AUC, confusion matrix |
| No usar `stratify` en split | Distribución diferente en train/test | `train_test_split(..., stratify=y)` |

---
# PARTE 5 – Checklist para el día del parcial

Cuando te den el caso, antes de escribir código:

**Paso 1 – Identificar el tipo de problema**
- [ ] ¿Supervisado (hay etiquetas) o no supervisado (solo X)?
- [ ] ¿Clasificación (categorías) o regresión (número continuo) o clustering?
- [ ] ¿Binario (2 clases) o multiclase (N > 2)?
- [ ] ¿Qué tipo de datos? (imagen, texto, tabular)

**Paso 2 – Elegir la arquitectura**
- [ ] Imagen → CNN
- [ ] Texto con secuencia → Embedding + LSTM/GRU
- [ ] Texto con embeddings pre-entrenados → BERT + clasificador
- [ ] Tabular → MLP o Bayes o árbol
- [ ] Sin etiquetas → K-Means / DBSCAN / Jerárquico

**Paso 3 – Verificar el código**
- [ ] ¿La activación de la capa de salida es correcta?
- [ ] ¿La función de pérdida corresponde al problema?
- [ ] ¿Hay Flatten antes de Dense en una CNN?
- [ ] ¿Las imágenes están normalizadas (÷255)?
- [ ] ¿Hay data leakage (scaler fit antes del split)?
- [ ] ¿Las métricas son apropiadas para el dataset (¿está balanceado?)?
- [ ] ¿El texto fue tokenizado y convertido a secuencias antes de la red?
- [ ] ¿Los datos de clustering están escalados?
- [ ] ¿El número de clusters en K-Means se justificó?